# AI4AM poster structure example

MatterGen example with a free Wyckoff positional parameter and mixed substituted-relaxed match outcomes among the top 3 `sm_anon` and top 3 Wyckoff candidates.


In [ ]:
from __future__ import annotations

import gzip
import json
import sys
from collections.abc import Iterable
from importlib import resources
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import pymatgen.analysis.prototypes as prototypes
from IPython.display import Markdown, display
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatviz import structure_2d

import __main__

for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    notebooks_dir = path / "notebooks"
    if (notebooks_dir / "notebook_utils.py").exists():
        if str(notebooks_dir) not in sys.path:
            sys.path.insert(0, str(notebooks_dir))
        break
else:
    raise RuntimeError("Could not find notebooks directory")

from notebook_utils import find_repo_root  # noqa: E402

ROOT = find_repo_root()
NOTEBOOKS_DIR = ROOT / "notebooks"
SCRIPTS_DIR = ROOT / "scripts"
for import_path in (ROOT, NOTEBOOKS_DIR, SCRIPTS_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from notebook_constants import WYCKOFF_REPR_FILE  # noqa: E402
from notebook_utils import load_pickle_gz, required_paths  # noqa: E402
from substitute_structures import SubstitutedEntry  # noqa: E402

from src.config import INPUT_DIR, RESULTS_DIR  # noqa: E402
from src.sm_anon import AnonMatch  # noqa: E402

# Some substituted-entry pickles were written with SubstitutedEntry resolved
# from __main__. Provide that symbol before unpickling.
__main__.SubstitutedEntry = SubstitutedEntry

MODEL = "mattergen"
GEN_IDX = 1297
EXPECTED_SPG_NUM = 123
EXPECTED_CRYSTAL_SYSTEM = "tetragonal"
EXPECTED_CONVENTIONAL_ATOMS = 4
EXPECTED_FREE_WYCKOFF_LETTERS = {"h"}
STRUCTURE_OUT_DIR = RESULTS_DIR / "ai4am" / "poster_structures"

paths = required_paths(MODEL, INPUT_DIR, RESULTS_DIR)
WYCKOFF_PARAMS_PATH = resources.files(prototypes).joinpath(
    "wyckoff-position-params.json.gz"
)
with gzip.open(WYCKOFF_PARAMS_PATH, "rt") as file:
    WYCKOFF_POSITION_PARAMS = json.load(file)

In [ ]:
generated_structures: list[Structure] = load_pickle_gz(paths["generated_structures"])
training_structures: list[Structure] = load_pickle_gz(paths["training_structures"])
wyckoff_repr = load_pickle_gz(paths["wyckoff_repr"])
train_wyckoff_repr = load_pickle_gz(RESULTS_DIR / "train" / WYCKOFF_REPR_FILE)
wyckoff_matches = load_pickle_gz(paths["wyckoff_matches"])

raw_entries_by_source = {
    "sm_anon": load_pickle_gz(paths["top3_sm_anon"]),
    "wyckoff": load_pickle_gz(paths["top3_wyckoff"]),
}
entries_by_source = {
    "sm_anon": load_pickle_gz(paths["relaxed_sm_anon_entries"]),
    "wyckoff": load_pickle_gz(paths["relaxed_wyckoff_entries"]),
}
records_by_source = {
    "sm_anon": load_pickle_gz(paths["relaxed_sm_anon_matches"]),
    "wyckoff": load_pickle_gz(paths["relaxed_wyckoff_matches"]),
}
infos_by_source = {
    "sm_anon": load_pickle_gz(
        paths["relaxed_sm_anon_entries"].with_name(
            paths["relaxed_sm_anon_entries"].name.removesuffix(".pkl.gz")
            + "_infos.pkl.gz"
        )
    ),
    "wyckoff": load_pickle_gz(
        paths["relaxed_wyckoff_entries"].with_name(
            paths["relaxed_wyckoff_entries"].name.removesuffix(".pkl.gz")
            + "_infos.pkl.gz"
        )
    ),
}


def sm_anon_match_lookup(
    gen_idx: int,
    train_indices: Iterable[int],
) -> dict[tuple[int, int], AnonMatch]:
    wanted = {(gen_idx, int(train_idx)) for train_idx in train_indices}
    found: dict[tuple[int, int], AnonMatch] = {}
    for match in load_pickle_gz(paths["sm_anon_matches"]):
        key = (int(match.idx1), int(match.idx2))
        if key in wanted:
            found[key] = match
            if len(found) == len(wanted):
                break
    missing = wanted - found.keys()
    if missing:
        raise ValueError(f"Missing sm_anon matches for {sorted(missing)}")
    return found


def crystal_system_from_spg_num(spg_num: int) -> str:
    match spg_num:
        case n if 16 <= n <= 74:
            return "orthorhombic"
        case n if 75 <= n <= 142:
            return "tetragonal"
        case n if 195 <= n <= 230:
            return "cubic"
        case _:
            return "other"


def generated_wyckoff_rows(structure: Structure) -> pd.DataFrame:
    symmetrized = SpacegroupAnalyzer(
        structure, symprec=0.01
    ).get_symmetrized_structure()
    rows = []
    for wyckoff_symbol, sites_group in zip(
        symmetrized.wyckoff_symbols,
        symmetrized.equivalent_sites,
        strict=True,
    ):
        letter = wyckoff_symbol.lstrip("0123456789")
        n_free = int(WYCKOFF_POSITION_PARAMS[str(spg_num)].get(letter, 0))
        rows.append(
            {
                "wyckoff_symbol": wyckoff_symbol,
                "letter": letter,
                "multiplicity_in_cell": len(sites_group),
                "element": sites_group[0].species_string,
                "n_free_parameters": n_free,
                "representative_frac_coords": tuple(
                    float(value) for value in sites_group[0].frac_coords
                ),
            }
        )
    return pd.DataFrame(rows)


def conventional_structure(structure: Structure) -> Structure:
    return SpacegroupAnalyzer(
        structure,
        symprec=0.01,
    ).get_conventional_standard_structure()


def safe_label(value: str) -> str:
    return "".join(char if char.isalnum() else "_" for char in value)


def save_conventional_cif(
    structure: Structure,
    filename: str,
) -> tuple[Structure, Path]:
    conventional = conventional_structure(structure)
    STRUCTURE_OUT_DIR.mkdir(parents=True, exist_ok=True)
    path = STRUCTURE_OUT_DIR / filename
    conventional.to(filename=path)
    return conventional, path


def highest_cost_wyckoff_rows(gen_idx: int, k: int = 3) -> pd.DataFrame:
    if len(train_wyckoff_repr) != len(training_structures):
        raise ValueError(
            "Training Wyckoff data length does not match training structures."
        )
    matches = [
        match
        for match in wyckoff_matches
        if int(match.idx1) == gen_idx and not np.isnan(float(match.cost_mod_petti))
    ]
    ranked = sorted(
        matches,
        key=lambda match: (-float(match.cost_mod_petti), int(match.idx2)),
    )[:k]
    if len(ranked) != k:
        raise ValueError(
            f"Expected {k} Wyckoff matches for gen_idx={gen_idx}; found {len(ranked)}."
        )

    rows: list[dict[str, Any]] = []
    for rank, match in enumerate(ranked, start=1):
        train_idx = int(match.idx2)
        training_structure = training_structures[train_idx]
        lattice = training_structure.lattice
        rows.append(
            {
                "rank": rank,
                "gen_idx": int(match.idx1),
                "train_idx": train_idx,
                "train_formula": training_structure.composition.reduced_formula,
                "train_spg_num": int(train_wyckoff_repr[train_idx].spg_num),
                "train_a": float(lattice.a),
                "train_b": float(lattice.b),
                "train_c": float(lattice.c),
                "train_alpha": float(lattice.alpha),
                "train_beta": float(lattice.beta),
                "train_gamma": float(lattice.gamma),
                "cost_uniform": float(match.cost_uniform),
                "cost_mod_petti": float(match.cost_mod_petti),
                "training_structure": training_structure,
            }
        )
    return pd.DataFrame(rows)


def record_rows(gen_idx: int) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    sm_anon_train_indices = [
        int(entry.train_idx)
        for entry in entries_by_source["sm_anon"]
        if int(entry.gen_idx) == gen_idx and int(entry.rank) <= 3
    ]
    sm_anon_matches = sm_anon_match_lookup(gen_idx, sm_anon_train_indices)
    if len(train_wyckoff_repr) != len(training_structures):
        raise ValueError(
            "Training Wyckoff data length does not match training structures."
        )
    for source in ("sm_anon", "wyckoff"):
        for entry_idx, (raw_entry, entry, record, info) in enumerate(
            zip(
                raw_entries_by_source[source],
                entries_by_source[source],
                records_by_source[source],
                infos_by_source[source],
                strict=True,
            )
        ):
            if int(entry.gen_idx) != gen_idx or int(entry.rank) > 3:
                continue
            if int(record["entry_idx"]) != entry_idx:
                raise ValueError(f"{source}: entry_idx mismatch at {entry_idx}")
            for attr in ("gen_idx", "train_idx", "rank"):
                if int(getattr(raw_entry, attr)) != int(getattr(entry, attr)):
                    raise ValueError(
                        f"{source}: raw/relaxed {attr} mismatch at {entry_idx}"
                    )
                if int(getattr(entry, attr)) != int(record[attr]):
                    raise ValueError(f"{source}: {attr} mismatch at {entry_idx}")
            for attr in ("cost_uniform", "cost_mod_petti"):
                if not np.isclose(
                    float(getattr(raw_entry, attr)), float(getattr(entry, attr))
                ):
                    raise ValueError(
                        f"{source}: raw/relaxed {attr} mismatch at {entry_idx}"
                    )
                if not np.isclose(float(getattr(entry, attr)), float(record[attr])):
                    raise ValueError(f"{source}: {attr} mismatch at {entry_idx}")
            training_structure = training_structures[int(entry.train_idx)]
            lattice = training_structure.lattice
            rows.append(
                {
                    "source": source,
                    "rank": int(entry.rank),
                    "entry_idx": entry_idx,
                    "gen_idx": int(entry.gen_idx),
                    "train_idx": int(entry.train_idx),
                    "train_formula": training_structure.composition.reduced_formula,
                    "train_spg_num": int(
                        train_wyckoff_repr[int(entry.train_idx)].spg_num
                    ),
                    "train_a": float(lattice.a),
                    "train_b": float(lattice.b),
                    "train_c": float(lattice.c),
                    "train_alpha": float(lattice.alpha),
                    "train_beta": float(lattice.beta),
                    "train_gamma": float(lattice.gamma),
                    "cost_uniform": float(entry.cost_uniform),
                    "cost_mod_petti": float(entry.cost_mod_petti),
                    "relaxed_match": bool(record["match"]),
                    "relax_failed": info is None,
                    "relax_converged": None
                    if info is None
                    else bool(info.get("converged", False)),
                    "sm_anon_match": sm_anon_matches[
                        (int(entry.gen_idx), int(entry.train_idx))
                    ]
                    if source == "sm_anon"
                    else None,
                    "training_structure": training_structure,
                    "substituted_structure": raw_entry.structure,
                    "relaxed_substituted_structure": entry.structure,
                }
            )
    frame = pd.DataFrame(rows)
    return frame.sort_values(["source", "rank"]).reset_index(drop=True)


example = record_rows(GEN_IDX)
highest_cost_wyckoff = highest_cost_wyckoff_rows(GEN_IDX)
generated_structure = generated_structures[GEN_IDX]
spg_num = int(wyckoff_repr[GEN_IDX].spg_num)
crystal_system = crystal_system_from_spg_num(spg_num)

assert spg_num == EXPECTED_SPG_NUM
assert crystal_system == EXPECTED_CRYSTAL_SYSTEM
generated_wyckoff = generated_wyckoff_rows(generated_structure)
free_wyckoff_letters = set(
    generated_wyckoff.loc[
        generated_wyckoff["n_free_parameters"] > 0,
        "letter",
    ]
)
generated_conventional_preview = conventional_structure(generated_structure)

assert len(generated_conventional_preview) == EXPECTED_CONVENTIONAL_ATOMS
assert EXPECTED_FREE_WYCKOFF_LETTERS <= free_wyckoff_letters
assert len(example) == 6
assert len(highest_cost_wyckoff) == 3
assert highest_cost_wyckoff["cost_mod_petti"].is_monotonic_decreasing
assert example["relaxed_match"].any()
assert (~example["relaxed_match"] | example["relax_failed"]).any()

save_records: list[dict[str, Any]] = []
generated_conventional_structure, generated_cif_path = save_conventional_cif(
    generated_structure,
    f"mattergen_gen_{GEN_IDX}_{safe_label(generated_structure.composition.reduced_formula)}"
    "_conventional.cif",
)
save_records.append(
    {
        "role": "generated",
        "source": None,
        "rank": None,
        "index": GEN_IDX,
        "formula": generated_structure.composition.reduced_formula,
        "path": generated_cif_path,
    }
)

training_conventional_structures = []
substituted_conventional_structures = []
relaxed_conventional_structures = []
training_cif_paths = []
substituted_cif_paths = []
relaxed_cif_paths = []
for row in example.itertuples(index=False):
    key = f"{row.source}_rank{int(row.rank)}_train_{int(row.train_idx)}"
    formula = safe_label(row.train_formula)
    training_conventional, training_cif_path = save_conventional_cif(
        row.training_structure,
        f"{key}_{formula}_training_conventional.cif",
    )
    substituted_conventional, substituted_cif_path = save_conventional_cif(
        row.substituted_structure,
        f"{key}_{formula}_substituted_conventional.cif",
    )
    relaxed_conventional, relaxed_cif_path = save_conventional_cif(
        row.relaxed_substituted_structure,
        f"{key}_{formula}_relaxed_substituted_conventional.cif",
    )
    training_conventional_structures.append(training_conventional)
    substituted_conventional_structures.append(substituted_conventional)
    relaxed_conventional_structures.append(relaxed_conventional)
    training_cif_paths.append(training_cif_path)
    substituted_cif_paths.append(substituted_cif_path)
    relaxed_cif_paths.append(relaxed_cif_path)
    save_records.extend(
        [
            {
                "role": "training",
                "source": row.source,
                "rank": int(row.rank),
                "index": int(row.train_idx),
                "formula": row.train_formula,
                "path": training_cif_path,
            },
            {
                "role": "substituted",
                "source": row.source,
                "rank": int(row.rank),
                "index": int(row.train_idx),
                "formula": row.train_formula,
                "path": substituted_cif_path,
            },
            {
                "role": "relaxed_substituted",
                "source": row.source,
                "rank": int(row.rank),
                "index": int(row.train_idx),
                "formula": row.train_formula,
                "path": relaxed_cif_path,
            },
        ]
    )

example = example.assign(
    training_conventional_structure=training_conventional_structures,
    substituted_conventional_structure=substituted_conventional_structures,
    relaxed_substituted_conventional_structure=relaxed_conventional_structures,
    training_conventional_cif=training_cif_paths,
    substituted_conventional_cif=substituted_cif_paths,
    relaxed_substituted_conventional_cif=relaxed_cif_paths,
)

highest_cost_training_conventional_structures = []
highest_cost_training_cif_paths = []
for row in highest_cost_wyckoff.itertuples(index=False):
    formula = safe_label(row.train_formula)
    training_conventional, training_cif_path = save_conventional_cif(
        row.training_structure,
        f"wyckoff_highest_cost_rank{int(row.rank)}_train_{int(row.train_idx)}"
        f"_{formula}_training_conventional.cif",
    )
    highest_cost_training_conventional_structures.append(training_conventional)
    highest_cost_training_cif_paths.append(training_cif_path)
    save_records.append(
        {
            "role": "highest_cost_wyckoff_training",
            "source": "wyckoff",
            "rank": int(row.rank),
            "index": int(row.train_idx),
            "formula": row.train_formula,
            "path": training_cif_path,
        }
    )

highest_cost_wyckoff = highest_cost_wyckoff.assign(
    training_conventional_structure=highest_cost_training_conventional_structures,
    training_conventional_cif=highest_cost_training_cif_paths,
)
saved_cifs = pd.DataFrame(save_records)

display(
    Markdown(
        f"## Selected MatterGen example  \\n"
        f"gen_idx=`{GEN_IDX}`; "
        f"formula=`{generated_structure.composition.reduced_formula}`; "
        f"conventional atoms=`{len(generated_conventional_structure)}`; "
        f"space group=`{spg_num}`; crystal system=`{crystal_system}`; "
        f"free Wyckoff letters=`{sorted(free_wyckoff_letters)}`"
    )
)
display(generated_wyckoff)
display(Markdown("## Highest-cost Wyckoff training matches"))
display(
    highest_cost_wyckoff.drop(
        columns=["training_structure", "training_conventional_structure"]
    )
)
display(
    example.drop(
        columns=[
            "sm_anon_match",
            "training_structure",
            "substituted_structure",
            "relaxed_substituted_structure",
            "training_conventional_structure",
            "substituted_conventional_structure",
            "relaxed_substituted_conventional_structure",
        ]
    )
)
display(Markdown(f"## Conventional CIFs saved to `{STRUCTURE_OUT_DIR}`"))

In [ ]:
def status_label(row: pd.Series) -> str:
    if row["relax_failed"]:
        return "relax failed"
    if row["relaxed_match"]:
        return "relaxed match"
    return "no relaxed match"


def format_cost(value: Any) -> str:
    if pd.isna(value):
        return "N/A"
    return f"{float(value):.6g}"


def panel_title(row: pd.Series, role: str) -> str:
    return (
        f"{role}<br>{row['source']} rank {int(row['rank'])}; "
        f"train_idx={int(row['train_idx'])}<br>"
        f"{row['train_formula']}; {status_label(row)}<br>"
        f"uniform={format_cost(row['cost_uniform'])}; "
        f"mod-Petti={format_cost(row['cost_mod_petti'])}"
    )


generated_fig = structure_2d(
    {"generated": generated_structure},
    show_cell=True,
    standardize_struct=False,
    subplot_title=lambda _struct, _key: (
        f"MatterGen generated<br>gen_idx={GEN_IDX}; "
        f"{generated_structure.composition.reduced_formula}<br>"
        f"spg={spg_num}; {crystal_system}"
    ),
)
generated_fig.update_layout(height=320, margin={"l": 10, "r": 10, "t": 80, "b": 10})
display(generated_fig)

train_structures = {
    f"{row.source}_{int(row.rank)}": row.training_structure
    for row in example.itertuples(index=False)
}
train_titles = {
    f"{row.source}_{int(row.rank)}": panel_title(pd.Series(row._asdict()), "Training")
    for row in example.itertuples(index=False)
}
train_fig = structure_2d(
    train_structures,
    n_cols=3,
    show_cell=True,
    standardize_struct=False,
    subplot_title=lambda _struct, key: train_titles[key],
)
train_fig.update_layout(height=720, margin={"l": 10, "r": 10, "t": 90, "b": 10})
display(Markdown("## Matched training structures"))
display(train_fig)


def highest_cost_wyckoff_title(row: pd.Series) -> str:
    return (
        f"Wyckoff highest-cost rank {int(row['rank'])}<br>"
        f"train_idx={int(row['train_idx'])}<br>"
        f"{row['train_formula']}<br>"
        f"uniform={format_cost(row['cost_uniform'])}; "
        f"mod-Petti={format_cost(row['cost_mod_petti'])}"
    )


highest_cost_wyckoff_structures = {
    f"wyckoff_high_cost_{int(row.rank)}": row.training_structure
    for row in highest_cost_wyckoff.itertuples(index=False)
}
highest_cost_wyckoff_titles = {
    f"wyckoff_high_cost_{int(row.rank)}": highest_cost_wyckoff_title(
        pd.Series(row._asdict())
    )
    for row in highest_cost_wyckoff.itertuples(index=False)
}
highest_cost_wyckoff_fig = structure_2d(
    highest_cost_wyckoff_structures,
    n_cols=3,
    show_cell=True,
    standardize_struct=False,
    subplot_title=lambda _struct, key: highest_cost_wyckoff_titles[key],
)
highest_cost_wyckoff_fig.update_layout(
    height=420,
    margin={"l": 10, "r": 10, "t": 90, "b": 10},
)
display(Markdown("## Highest-cost Wyckoff training structures"))
display(highest_cost_wyckoff_fig)

relaxed_structures = {
    f"{row.source}_{int(row.rank)}": row.relaxed_substituted_conventional_structure
    for row in example.itertuples(index=False)
}
relaxed_titles = {
    f"{row.source}_{int(row.rank)}": panel_title(
        pd.Series(row._asdict()), "Relaxed substituted"
    )
    for row in example.itertuples(index=False)
}
relaxed_fig = structure_2d(
    relaxed_structures,
    n_cols=3,
    show_cell=True,
    standardize_struct=False,
    subplot_title=lambda _struct, key: relaxed_titles[key],
)
relaxed_fig.update_layout(height=720, margin={"l": 10, "r": 10, "t": 90, "b": 10})
display(Markdown("## Relaxed substituted training structures"))
display(relaxed_fig)


def sm_anon_commensurate_structures(row: pd.Series) -> tuple[Structure, Structure]:
    match = row["sm_anon_match"]
    if match is None:
        raise ValueError("sm_anon_match is required")

    generated = generated_structures[int(row["gen_idx"])].copy()
    training = row["training_structure"].copy()
    if bool(match.s1_supercell):
        generated.make_supercell(match.supercell_matrix)
    else:
        training.make_supercell(match.supercell_matrix)

    if len(generated) != len(training):
        raise ValueError(
            "sm_anon commensurate generated/training structures have different sizes."
        )
    if len(match.mapping) != len(generated):
        raise ValueError(
            "sm_anon mapping length does not match commensurate structure size."
        )
    if min(match.mapping) < 0 or max(match.mapping) >= len(generated):
        raise ValueError("sm_anon mapping contains an out-of-bounds atom index.")
    return generated, training


sm_anon_rows = example[example["source"] == "sm_anon"].sort_values("rank")
generated_commensurate_structures = {}
generated_commensurate_titles = {}
for row in sm_anon_rows.itertuples(index=False):
    row_series = pd.Series(row._asdict())
    key = f"sm_anon_{int(row.rank)}"
    generated_commensurate, _training_commensurate = sm_anon_commensurate_structures(
        row_series
    )
    generated_commensurate_structures[key] = generated_commensurate
    generated_commensurate_titles[key] = (
        f"Generated commensurate cell<br>sm_anon rank {int(row.rank)}; "
        f"train_idx={int(row.train_idx)}"
    )

generated_supercell_fig = structure_2d(
    generated_commensurate_structures,
    n_cols=3,
    show_cell=True,
    standardize_struct=False,
    subplot_title=lambda _struct, key: generated_commensurate_titles[key],
)
generated_supercell_fig.update_layout(
    height=420,
    margin={"l": 10, "r": 10, "t": 90, "b": 10},
)
display(Markdown("## Generated commensurate cells used by sm_anon"))
display(generated_supercell_fig)